In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Load all datasets
customers = pd.read_csv("../src_data/olist_customers_dataset.csv")
orders = pd.read_csv("../src_data/olist_orders_dataset.csv")
order_items = pd.read_csv("../src_data/olist_order_items_dataset.csv")
products = pd.read_csv("../src_data/olist_products_dataset.csv")
sellers = pd.read_csv("../src_data/olist_sellers_dataset.csv")
payments = pd.read_csv("../src_data/olist_order_payments_dataset.csv")
geo = pd.read_csv("../src_data/olist_geolocation_dataset.csv")
category = pd.read_csv("../src_data/product_category_name_translation.csv")

In [3]:
# Merge orders with customers to get customer details in the orders dataframe
df = orders.copy()
df = df.merge(customers, on="customer_id", how="left")

In [4]:
# Merge orders with order_items to get product details in the orders dataframe
df = df.merge(order_items, on="order_id", how="left")
# Merge orders with products to get product details
df = df.merge(products, on="product_id", how="left")
# Merge orders with category to get category details
df = df.merge(category, on="product_category_name", how="left")
# Merge orders with sellers to get seller details
df = df.merge(sellers, on="seller_id", how="left")
# Merge orders with payments to get payment details
df = df.merge(payments, on="order_id", how="left")

In [5]:
# Aggregate geolocation data to get average latitude and longitude for each zip code prefix
geography = geo.groupby("geolocation_zip_code_prefix").agg({
    "geolocation_lat": "mean",
    "geolocation_lng": "mean"
}).reset_index()

In [6]:
# Merge geolocation data with the main dataframe to get customer latitude and longitude based on their zip code prefix
df = df.merge(
    geography,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
).rename(columns={
    "geolocation_lat": "customer_lat",
    "geolocation_lng": "customer_lng"
}).drop(columns=["geolocation_zip_code_prefix"])

In [7]:
# Merge geolocation data with the main dataframe to get seller latitude and longitude based on their zip code prefix
df = df.merge(
    geography,
    left_on="seller_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
).rename(columns={
    "geolocation_lat": "seller_lat",
    "geolocation_lng": "seller_lng"
}).drop(columns=["geolocation_zip_code_prefix"])

In [8]:
df_final = df.groupby("order_id").agg({
    "price": "sum",
    "freight_value": "sum",
    "payment_value": "sum",
    "payment_installments": "max",
    "product_weight_g": "mean",
    
    "customer_state": "first",
    "seller_state": "first",
    
    "customer_lat": "first",
    "customer_lng": "first",
    "seller_lat": "first",
    "seller_lng": "first",
    
    "order_purchase_timestamp": "first",
    "order_approved_at": "first",
    "order_delivered_carrier_date": "first",
    "order_estimated_delivery_date": "first",
    "order_delivered_customer_date": "first"
}).reset_index()

Why use "first"? Because these details apply to the whole order and generally don't change from item to item. An order only has one purchase time and one delivery destination. Telling Pandas to Keep this data as it is; don't try to add it up.

In [9]:
# Calculate delivery time in days (Target variable)
df["delivery_time_days"] = (pd.to_datetime(df["order_estimated_delivery_date"]) - pd.to_datetime(df["order_purchase_timestamp"])).dt.days

In [10]:
# Convert date columns to datetime format
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    df[column] = pd.to_datetime(df[column])

In [11]:
# Calculate approval time in hours
df["approval_hours"] = (df["order_approved_at"] - df["order_purchase_timestamp"]).dt.total_seconds() / 3600

In [12]:
# Calculate carrier handover time in hours
df["carrier_handover_hours"] = (df["order_delivered_carrier_date"] -df["order_approved_at"]).dt.total_seconds() / 3600

In [13]:
# Extract purchase month as a feature
df["purchase_month"] = df["order_purchase_timestamp"].dt.month

In [14]:
# Extract purchase day of week as a feature
df["purchase_dayofweek"] = df["order_purchase_timestamp"].dt.dayofweek

In [15]:
# Handle missing values in numerical columns by filling with median
numerical_columns = [
    "price",
    "freight_value",
    "payment_value",
    "payment_installments",
    "product_weight_g",
    "approval_hours",
    "carrier_handover_hours"
]

for column in numerical_columns:
    df[column] = df[column].fillna(df[column].median())

In [16]:
# Handle missing values in categorical columns by filling with "Unknown"
df["customer_state"] = df["customer_state"].fillna("Unknown")
df["seller_state"] = df["seller_state"].fillna("Unknown")

In [17]:
# Handle missing values in geographical coordinates by filling with median
numerical_columns = [
    'product_photos_qty',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm',
    'customer_lat',
    'customer_lng',
    'seller_lat',
    'seller_lng'
]

for column in numerical_columns:
    df[column] = df[column].fillna(df[column].median())

In [18]:
# Handle missing values in categorical columns by filling with "Unknown" or mode
df = df.fillna({
    'product_category_name_english': 'Unknown',
    'payment_type': df['payment_type'].mode()[0]
})

In [19]:
df.isnull().sum()   

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 176
order_delivered_carrier_date     2074
order_delivered_customer_date    3397
order_estimated_delivery_date       0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
order_item_id                     830
product_id                        830
seller_id                         830
shipping_limit_date               830
price                               0
freight_value                       0
product_category_name            2528
product_name_lenght              2528
product_description_lenght       2528
product_photos_qty                  0
product_weight_g                    0
product_length_cm                   0
product_height_cm                   0
product_width_cm                    0
product_cate

In [20]:
df.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value',
       'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'product_category_name_english', 'seller_zip_code_prefix',
       'seller_city', 'seller_state', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value', 'customer_lat', 'customer_lng',
       'seller_lat', 'seller_lng', 'delivery_time_days', 'approval_hours',
       'carrier_handover_hours', 'purchase_month', 'purchase_dayofweek'],
      dtype='str

In [21]:
# Features to drop
drop_columns = [
    'customer_unique_id',
    'product_id',
    'seller_id',
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'shipping_limit_date',
    'customer_city',
    'seller_city',
    'customer_zip_code_prefix',
    'seller_zip_code_prefix',
    'product_category_name',
    'payment_sequential', 
    'order_item_id',
    'product_name_lenght',
    'product_description_lenght'
]

df = df.drop(columns=[column for column in drop_columns if column in df.columns])
print(df.columns.tolist())
print(f"\nRemaining Features: {df.shape[1]}")

['order_id', 'customer_id', 'order_status', 'customer_state', 'price', 'freight_value', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'seller_state', 'payment_type', 'payment_installments', 'payment_value', 'customer_lat', 'customer_lng', 'seller_lat', 'seller_lng', 'delivery_time_days', 'approval_hours', 'carrier_handover_hours', 'purchase_month', 'purchase_dayofweek']

Remaining Features: 25


In [22]:
data = df.to_csv("../data/raw_data.csv", index=False)